# Exact eight-team instructor evaluator on Google Colab

This notebook measures the current production evaluator, not the price-taking screening approximation. Upload the companion `ercot_colab_instructor_evaluator.zip` when prompted. Run the 24-hour smoke test first, then the full 4,368-hour test.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil, zipfile

uploaded = files.upload()
zip_candidates = [Path(name) for name in uploaded if name.endswith('.zip')]
if len(zip_candidates) != 1:
    raise RuntimeError('Upload exactly one ercot_colab_instructor_evaluator.zip file.')
bundle_zip = zip_candidates[0]
bundle_root = Path('/content/ercot_colab_instructor_evaluator')
if bundle_root.exists():
    shutil.rmtree(bundle_root)
with zipfile.ZipFile(bundle_zip) as archive:
    archive.extractall('/content')
if not (bundle_root / 'run_colab_evaluator.py').is_file():
    raise RuntimeError(f'Unexpected archive layout: {bundle_root}')
print('Bundle ready:', bundle_root)

In [ ]:
import subprocess, sys
root = Path('/content/ercot_colab_instructor_evaluator')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root / 'requirements-colab.txt')], check=True)
print('Dependencies installed.')

## 24-hour smoke test

This checks the archive, CSVs, rules, imports, and exact evaluator wiring before starting the long run.

In [ ]:
smoke = root / 'results' / 'smoke_24h'
subprocess.run([sys.executable, str(root / 'run_colab_evaluator.py'), '--horizon', '24', '--workers', '4', '--solver-backend', 'auto', '--output-dir', str(smoke)], check=True)

## Full 4,368-hour timing test

This is the same eight-team batch evaluation configuration used for the instructor example. Colab will usually resolve `auto` to HiGHS unless a usable Gurobi installation and license are explicitly configured.

In [ ]:
full = root / 'results' / 'full_4368h'
subprocess.run([sys.executable, str(root / 'run_colab_evaluator.py'), '--horizon', '4368', '--workers', '4', '--solver-backend', 'auto', '--output-dir', str(full)], check=True)

In [ ]:
import json
summary_path = full / 'colab_timing_summary.json'
print(json.dumps(json.loads(summary_path.read_text()), indent=2))
print('Download the summary and compact evaluation output if needed:')
print(summary_path)
print(full / 'evaluation.json')